# Are Longer Books Rated Higher on Goodreads?
**CIS 605 Programming for Data Science · Project · Fall 2026**

## 1. Dataset
**Name:** Goodreads-books
**Source:** Kaggle, uploaded by Soumik: https://www.kaggle.com/datasets/jealousleopard/goodreadsbooks
**File:** books.csv (1.56 MB), more than 11,000 rows and 12 columns

Goodreads is a website where readers rate and review the books they read. Each row in this dataset is one book edition listed on Goodreads. For every book it gives the title, the authors, the average reader rating, how many ratings and written reviews it received, its number of pages, its language, its publication date and its publisher.

### Research question
**Is the length of a book associated with its average rating on Goodreads?**

To answer it, I will compare the average rating of short, medium and long books, measure the correlation between the number of pages and the average rating, and look at how ratings are spread within each length group. The analysis describes associations only. It does not claim that the length of a book causes a higher or lower rating.

### Data dictionary
| Column | Description | Type | Role in the analysis |
|---|---|---|---|
| title | Title of the book as published | Text | Identifying books |
| authors | Author name(s); several authors are separated by "/" | Text | Identifying books |
| average_rating | Average reader rating on Goodreads, from 0 to 5 | Number (decimal) | Outcome: the rating we compare |
| num_pages | Number of pages in the edition | Number (integer) | Main variable: book length |
| ratings_count | Total number of ratings the book received | Number (integer) | Cleaning: a book with no ratings has no real average rating |

The other seven columns (bookID, isbn, isbn13, language_code, text_reviews_count, publication_date, publisher) are not needed to answer the research question.

## 2. Data Inspection and Preparation

### 2.1 Loading the data
I loaded books.csv with Pandas straight from my GitHub repository, so the notebook runs on any computer without downloading the file first. I kept this first table under the name `books_raw` and I do not change it, so the original data stays as it is. A few rows in the file have more fields than the 12 columns, which stops Pandas from reading the file, so I used `on_bad_lines="skip"` to leave them out. I then printed the shape, the column names, the data types and the summary statistics to get a first look at the data.

In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/salambaniateah-afk/goodreads-project/main/books.csv"
books_raw = pd.read_csv(url, on_bad_lines="skip")
print(books_raw.shape)
print(books_raw.columns)
print(books_raw.dtypes)
books_raw.describe()

(11123, 12)
Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher'],
      dtype='str')
bookID                  int64
title                     str
authors                   str
average_rating        float64
isbn                      str
isbn13                  int64
language_code             str
  num_pages             int64
ratings_count           int64
text_reviews_count      int64
publication_date          str
publisher                 str
dtype: object


,bookID,average_rating,isbn13,num_pages,ratings_count,text_reviews_count
count,11123.000000,11123.000000,1.112300e+04,11123.000000,1.112300e+04,11123.000000
mean,21310.856963,3.934075,9.759880e+12,336.405556,1.794285e+04,542.048099
std,13094.727252,0.350485,4.429758e+11,241.152626,1.124992e+05,2576.619589
min,1.000000,0.000000,8.987060e+09,0.000000,0.000000e+00,0.000000
25%,10277.500000,3.770000,9.780345e+12,192.000000,1.040000e+02,9.000000
50%,20287.000000,3.960000,9.780582e+12,299.000000,7.450000e+02,47.000000
75%,32104.500000,4.140000,9.780872e+12,416.000000,5.000500e+03,238.000000
max,45641.000000,5.000000,9.790008e+12,6576.000000,4.597666e+06,94265.000000


In [4]:
from urllib.request import urlopen

total_rows = len(urlopen(url).read().decode("utf-8").splitlines()) - 1
print("Rows in the file:", total_rows)
print("Rows skipped:", total_rows - len(books_raw))
books = books_raw.copy()
books.columns = books.columns.str.strip()
print(books.columns)

Rows in the file: 11127
Rows skipped: 4
Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', 'num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher'],
      dtype='str')


### 2.2 First look at the data
After loading, the data has 11,123 rows and 12 columns. 4 rows were skipped because they had too many fields. The three columns I need for the analysis already have the right types: average_rating is a decimal number, and num_pages and ratings_count are whole numbers, so no type conversion is needed.

Two things stood out. First, the name of the num_pages column starts with spaces in the file ("num_pages"), so I made a working copy called `books` and removed the extra spaces from all column names. Second, the summary statistics show values that need checking. Some books have 0 pages, some have an average rating of 0, and some have 0 ratings. The longest book has 6,576 pages, while the median is 299 pages. Most ratings sit close to 4: the mean is 3.93, the median is 3.96 and the standard deviation is only 0.35. I check the unusual values in the next steps.

### 2.3 Missing values and duplicates
I counted the empty cells in every column and found none, so no missing values needed to be filled or removed. There are also no fully repeated rows and no repeated bookID values. However, 311 rows share the same title and authors as another row. These are different editions of the same book, for example a hardcover and a paperback, each listed as its own row. If I keep all of them, books with many editions would count more than once in the comparison. I deal with this in the cleaning step by keeping one row for each book.

Looking at some examples, all editions of one book share the same average rating. For example, every edition of 'Salem's Lot by Stephen King has a rating of 4.02. Their page counts are different, though, from 405 to 817 pages, and one edition lists 0 pages. So the rating belongs to the book, while the page count belongs to the edition. For each book I keep the edition with the most ratings, because it is the edition most readers used.

Missing value:

In [5]:
print(books.isnull().sum())
print("Duplicate rows:", books.duplicated().sum())
print("Duplicate bookIDs:", books["bookID"].duplicated().sum())
print("Same title and authors:", books.duplicated(subset=["title", "authors"]).sum())

bookID                0
title                 0
authors               0
average_rating        0
isbn                  0
isbn13                0
language_code         0
num_pages             0
ratings_count         0
text_reviews_count    0
publication_date      0
publisher             0
dtype: int64
Duplicate rows: 0
Duplicate bookIDs: 0
Same title and authors: 311


In [6]:
same_book = books[books.duplicated(subset=["title","authors"], keep=False)]
same_book.sort_values(["title","authors"])[["title","authors","average_rating","num_pages","ratings_count"]].head(10)

,title,authors,average_rating,num_pages,ratings_count
1574,'Salem's Lot,Stephen King,4.02,405,1039
1576,'Salem's Lot,Stephen King,4.02,427,178
1577,'Salem's Lot,Stephen King,4.02,586,25
5300,'Salem's Lot,Stephen King,4.02,817,18
9249,'Salem's Lot,Stephen King,4.02,427,186
1573,'Salem's Lot,Stephen King/Ron McLarty,4.02,17,227
5298,'Salem's Lot,Stephen King/Ron McLarty,4.02,0,56
9130,A Hard Day's Write: The Stories Behind Every B...,Steve Turner,4.18,224,156
9142,A Hard Day's Write: The Stories Behind Every B...,Steve Turner,4.18,224,2072
1164,A History of God: The 4 000-Year Quest of Juda...,Karen Armstrong,3.87,460,38915


In [7]:
print("Books with 0 pages:", (books["num_pages"] == 0).sum())
print("Books with 0 ratings:", (books["ratings_count"] == 0).sum())
print("Books with average_rating 0:", (books["average_rating"] == 0).sum())
print("Books with fewer than 10 ratings:", (books["ratings_count"] < 10).sum())
print("Books over 1500 pages:", (books["num_pages"] > 1500).sum())
books[books["num_pages"] > 1500][["title", "num_pages"]].head(10)

Books with 0 pages: 76
Books with 0 ratings: 80
Books with average_rating 0: 25
Books with fewer than 10 ratings: 673
Books over 1500 pages: 29


,title,num_pages
4,Harry Potter Boxed Set Books 1-5 (Harry Potte...,2690
6,Harry Potter Collection (Harry Potter #1-6),3342
21,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,1728
406,The Iliad/The Odyssey,1556
420,The Complete Pelican Shakespeare,1808
722,The History of the Lord of the Rings (The Hist...,1680
1053,The Complete Anne of Green Gables Boxed Set (A...,2088
1056,The New Annotated Sherlock Holmes: The Complet...,1878
2566,Plato: Complete Works,1838
3182,Rabbit Angstrom: The Four Novels,1520


### 2.4 Invalid and unusual values
Next I looked for values that cannot be right or that would distort the comparison. I found four problems:
- 76 books have 0 pages. A book cannot have zero pages, so the real length of these books is unknown.
- 80 books have 0 ratings, and 25 books have an average rating of 0. A rating of 0 here does not mean readers disliked the book; it means nobody rated it.
- 673 books have fewer than 10 ratings. Their average depends on very few readers, so one or two people can push it up or down a lot.
- 29 books have more than 1,500 pages. When I listed them, most were box sets and complete collections, such as Harry Potter Boxed Set Books 1-5 (2,690 pages) and Plato: Complete Works (1,838 pages). These are several books sold together, not one long book.

In [8]:
print("Rows before cleaning:", len(books))

books_clean = books[books["num_pages"] > 0]
print("After removing books with 0 pages:", len(books_clean))

books_clean = books_clean[books_clean["ratings_count"] >= 10]
print("After removing books with fewer than 10 ratings:", len(books_clean))

books_clean = books_clean[books_clean["num_pages"] <= 1500]
print("After removing books over 1500 pages:", len(books_clean))

books_clean = books_clean.sort_values("ratings_count", ascending=False)
books_clean = books_clean.drop_duplicates(subset=["title", "authors"], keep="first")
print("After keeping one edition per book:", len(books_clean))

books_clean = books_clean[["title","authors","average_rating","num_pages","ratings_count"]].reset_index(drop=True)
books_clean.describe()

Rows before cleaning: 11123
After removing books with 0 pages: 11047
After removing books with fewer than 10 ratings: 10395
After removing books over 1500 pages: 10367
After keeping one edition per book: 10083


,average_rating,num_pages,ratings_count
count,10083.000000,10083.000000,1.008300e+04
mean,3.945635,336.255182,1.970083e+04
std,0.274713,210.290887,1.179749e+05
min,2.400000,1.000000,1.000000e+01
25%,3.780000,200.000000,1.710000e+02
50%,3.960000,304.000000,9.910000e+02
75%,4.130000,416.000000,6.058000e+03
max,4.910000,1500.000000,4.597666e+06


### 2.5 Cleaning the data
I cleaned a separate table called `books_clean`, so `books_raw` and `books` stay unchanged. I made the changes in this order and printed the number of rows after each one:
1. Removed the 76 books with 0 pages, because their length is unknown (11,123 to 11,047 rows).
2. Removed books with fewer than 10 ratings, because their average rating is not reliable. This also removed every book with 0 ratings or an average rating of 0 (11,047 to 10,395 rows).
3. Removed books with more than 1,500 pages, because most of them are box sets of several books (10,395 to 10,367 rows).
4. Kept one edition of each book, the one with the most ratings (10,367 to 10,083 rows).

I removed the invalid rows before the duplicate editions, so the edition I keep always has a real page count. Finally, I kept only the five columns from the data dictionary.

In total I removed 1,040 rows, about 9% of the data, which leaves 10,083 books for the analysis. After cleaning, the average rating ranges from 2.40 to 4.91, with a mean of 3.95 and a median of 3.96. The number of pages ranges from 1 to 1,500, with a median of 304 pages.